In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report
import torch 
from datasets import Dataset
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, pipeline, Trainer, TrainingArguments

# Загрузка данных

In [2]:
train_path = Path(
    r"C:\Users\iliam\Desktop\My first NLP project\data\output\train_data.csv"
)
test_path = Path(
    r"C:\Users\iliam\Desktop\My first NLP project\data\output\test_data.csv"
)

In [3]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_df.head()
df = pd.concat([train_df, test_df], axis=0)
df.columns = ["text", "labels"]
df.head()

,text,labels
0,Хорошее приложение,1
1,"ТВ-помойка людям за 120,оформление вырвиглазно...",0
2,По дате загрузки можно искать,3
3,Простота и скорость,1
4,Отличное приложение,1


In [7]:
df_dataset = Dataset.from_pandas(df)
df_dataset = df_dataset.class_encode_column("labels")

Stringifying the column:   0%|          | 0/29056 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/29056 [00:00<?, ? examples/s]

In [8]:
id2label = {0: "negative", 1: "positive", 2: "trash", 3: "neutral"}
label2id = {"negative": 0, "positive": 1, "trash": 2, "neutral": 3}

# Загрузка модели cointegrated/rubert-tiny2 и токенизация

In [20]:
# Модели необходимо передавать прямой и обратный словарь классов
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
model = AutoModelForSequenceClassification.from_pretrained("cointegrated/rubert-tiny2", num_labels=4, id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
def preprocess_fn(examples):
    return tokenizer(examples["text"], max_length=512, truncation=True)

tokenized_dataset = df_dataset.map(preprocess_fn, batched=True, batch_size=1000)


Map:   0%|          | 0/29056 [00:00<?, ? examples/s]

In [22]:
tokenized_train_test_dataset = Dataset.train_test_split(tokenized_dataset, test_size=0.3, stratify_by_column='labels', seed=42)
tokenized_train_dataset = tokenized_train_test_dataset["train"]
tokenized_test_dataset = tokenized_train_test_dataset["test"]
tokenized_train_valid_dataset = Dataset.train_test_split(tokenized_train_dataset, test_size=0.2, stratify_by_column='labels', seed=42)
tokenized_train_dataset = tokenized_train_valid_dataset['train']
tokenized_valid_dataset = tokenized_train_valid_dataset["test"]

In [23]:
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def metrics(result):
    predictions, labels = result[0], result[1]
    preds = np.argmax(predictions, axis=-1)
    results = {
        "accuracy": accuracy_score(
            labels,
            preds
        ),
        "precision": precision_score(
            labels,
            preds,
            average='weighted',
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            preds,
            average='weighted',
            zero_division=0
        ),
        "f1_score": f1_score(
            labels,
            preds,
            average='weighted',
            zero_division=0
        )
    }
    return results

In [24]:
training_args = TrainingArguments(
    output_dir="models/bert_checkpoints",
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    eval_strategy="epoch",          # в старых версиях: evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    save_total_limit=2,             # хранить только 2 последних чекпоинта
    logging_steps=50,
    fp16=torch.cuda.is_available(), # смешанная точность на GPU
    report_to="none",               # без wandb и подобных сервисов
)

trainer = Trainer(model=model, 
                  args=training_args, 
                  train_dataset=tokenized_train_dataset, 
                  eval_dataset=tokenized_valid_dataset, 
                  data_collator=collator,
                  compute_metrics = metrics)

In [25]:
# Обучение
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Score
1,0.584830,0.548235,0.810718,0.808371,0.810718,0.794004
2,0.438186,0.447724,0.849312,0.846049,0.849312,0.846335
3,0.362539,0.414332,0.859145,0.856945,0.859145,0.856247
4,0.324202,0.404278,0.860374,0.857679,0.860374,0.857597
5,0.312633,0.401589,0.859145,0.856541,0.859145,0.856651


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2545, training_loss=0.46562102529529503, metrics={'train_runtime': 103.1097, 'train_samples_per_second': 789.014, 'train_steps_per_second': 24.682, 'total_flos': 125613477806880.0, 'train_loss': 0.46562102529529503, 'epoch': 5.0})

In [26]:
# Предсказания
predictions = trainer.predict(tokenized_test_dataset)

In [27]:
preds = np.argmax(predictions[0], axis=-1)
preds

array([0, 1, 0, ..., 1, 0, 1], shape=(8717,))

In [28]:
print(classification_report(predictions.label_ids, preds))

              precision    recall  f1-score   support

           0       0.87      0.94      0.90      3757
           1       0.88      0.89      0.88      2318
           2       0.91      0.83      0.87      1492
           3       0.75      0.61      0.67      1150

    accuracy                           0.86      8717
   macro avg       0.85      0.82      0.83      8717
weighted avg       0.86      0.86      0.86      8717



In [29]:
# Сохранение модели
trainer.save_model(r"C:\Users\iliam\Desktop\My first NLP project\models\bert_best")
tokenizer.save_pretrained(r"C:\Users\iliam\Desktop\My first NLP project\models\bert_best")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('C:\\Users\\iliam\\Desktop\\My first NLP project\\models\\bert_best\\tokenizer_config.json',
 'C:\\Users\\iliam\\Desktop\\My first NLP project\\models\\bert_best\\tokenizer.json')

In [ ]:
# Проверка
model_dir = Path.cwd().parent / "models" / "bert_best"
pipe = pipeline("text-classification", model=str(model_dir), device=0 if torch.cuda.is_available() else -1)
print(pipe.model.config.id2label)           # названия классов вместо LABEL_0
print(pipe(["Отличное приложение", "Ужасно, всё вылетает"]))

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

{0: 'negative', 1: 'positive', 2: 'trash', 3: 'neutral'}
[{'label': 'positive', 'score': 0.9796611070632935}, {'label': 'negative', 'score': 0.9788389801979065}]


## Загрузка модели Aniemore/rubert-large-emotion-russian-cedr-m7 и токенизация

In [5]:
tokenizer = AutoTokenizer.from_pretrained('Aniemore/rubert-large-emotion-russian-cedr-m7')
model = AutoModelForSequenceClassification.from_pretrained('Aniemore/rubert-large-emotion-russian-cedr-m7', 
                                                           num_labels=4, 
                                                           id2label=id2label, 
                                                           label2id=label2id, 
                                                           ignore_mismatched_sizes=True,
                                                           problem_type="single_label_classification")

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `7`.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: Aniemore/rubert-large-emotion-russian-cedr-m7
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7]) vs model:torch.Size([4])            
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7, 1024]) vs model:torch.Size([4, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [9]:
def preprocess_fn(examples):
    return tokenizer(examples["text"], max_length=512, truncation=True)
tokenized_dataset = df_dataset.map(preprocess_fn, batch_size=1000, batched=True)

Map:   0%|          | 0/29056 [00:00<?, ? examples/s]

In [10]:
tokenized_train_test_dataset = Dataset.train_test_split(tokenized_dataset, test_size=0.3, stratify_by_column='labels', seed=42)
tokenized_train_dataset = tokenized_train_test_dataset["train"]
tokenized_test_dataset = tokenized_train_test_dataset["test"]
tokenized_train_valid_dataset = Dataset.train_test_split(tokenized_train_dataset, test_size=0.2, stratify_by_column='labels', seed=42)
tokenized_train_dataset = tokenized_train_valid_dataset['train']
tokenized_valid_dataset = tokenized_train_valid_dataset["test"]

In [207]:
tokenized_train_dataset.shape

(16271, 6)

In [11]:
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def metrics(result):
    predictions, labels = result[0], result[1]
    preds = np.argmax(predictions, axis=-1)
    results = {
        "accuracy": accuracy_score(
            labels,
            preds
        ),
        "precision": precision_score(
            labels,
            preds,
            average='weighted',
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            preds,
            average='weighted',
            zero_division=0
        ),
        "f1_score": f1_score(
            labels,
            preds,
            average='weighted',
            zero_division=0
        )
    }
    return results

In [12]:
training_args = TrainingArguments(
    output_dir="models/bert_checkpoints",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    eval_strategy="epoch",          # в старых версиях: evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    save_total_limit=2,             # хранить только 2 последних чекпоинта
    logging_steps=50,
    fp16=torch.cuda.is_available(), # смешанная точность на GPU
    report_to="none",               # без wandb и подобных сервисов
)

trainer = Trainer(model=model, 
                  args=training_args, 
                  train_dataset=tokenized_train_dataset, 
                  eval_dataset=tokenized_valid_dataset, 
                  data_collator=collator,
                  compute_metrics = metrics)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Score
1,0.490417,0.418534,0.897247,0.896377,0.897247,0.893612
2,0.174226,0.509250,0.900688,0.903224,0.900688,0.901358
3,0.131851,0.552499,0.909046,0.908898,0.909046,0.908795


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12204, training_loss=0.3093860021088404, metrics={'train_runtime': 3180.5358, 'train_samples_per_second': 15.347, 'train_steps_per_second': 3.837, 'total_flos': 3966217473464928.0, 'train_loss': 0.3093860021088404, 'epoch': 3.0})

In [14]:
predictions = trainer.predict(tokenized_test_dataset)

In [16]:
preds = np.argmax(predictions[0], axis=-1)
print(classification_report(predictions.label_ids, preds))

              precision    recall  f1-score   support

           0       0.92      0.95      0.94      3757
           1       0.92      0.94      0.93      2318
           2       0.95      0.88      0.91      1492
           3       0.79      0.75      0.77      1150

    accuracy                           0.91      8717
   macro avg       0.90      0.88      0.89      8717
weighted avg       0.91      0.91      0.91      8717



In [19]:
# Сохранение модели
trainer.save_model(r"C:\Users\iliam\Desktop\My first NLP project\models\bert_best_2")
tokenizer.save_pretrained(r"C:\Users\iliam\Desktop\My first NLP project\models\bert_best_2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('C:\\Users\\iliam\\Desktop\\My first NLP project\\models\\bert_best_2\\tokenizer_config.json',
 'C:\\Users\\iliam\\Desktop\\My first NLP project\\models\\bert_best_2\\tokenizer.json')